# Text Classification using RNN and LSTM



In [2]:
!pip install -q tiktoken

import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import tiktoken

from google.colab import files
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)


PyTorch: 2.11.0+cu128
Device: cuda


## Rule 1 Load the dataset

The full CSV is loaded and 10,000 reviews are selected for lab training.


In [3]:
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)
print("Original dataset:", df.shape)

df = df[["review", "sentiment"]].dropna()

df = df.sample(
    n=10000,
    random_state=SEED
).reset_index(drop=True)

print("Working dataset:", df.shape)
display(df.head())
print("\nSentiment count:")
print(df["sentiment"].value_counts())


Saving IMDB Dataset(in).csv to IMDB Dataset(in).csv
Original dataset: (50000, 2)
Working dataset: (10000, 2)


,review,sentiment
0,I really liked this Summerslam due to the look...,positive
1,Not many television shows appeal to quite as m...,positive
2,The film quickly gets to a major chase scene w...,negative
3,Jane Austen would definitely approve of this o...,positive
4,Expectations were somewhat high for me when I ...,negative



Sentiment count:
sentiment
positive    5039
negative    4961
Name: count, dtype: int64


## Rule 2  Convert sentiment to numeric labels

Positive = 1, Negative = 0.


In [4]:
df["label"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
}).astype("int64")

print(df["label"].value_counts())


label
1    5039
0    4961
Name: count, dtype: int64


## Rule 3 Split into training and test sets

80% is used for training and 20% for testing.


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    df["review"],
    df["label"],
    test_size=0.20,
    random_state=SEED,
    stratify=df["label"]
)

X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))


Training samples: 8000
Testing samples : 2000


## Rule 4  Tokenize reviews with tiktoken

The required `cl100k_base` encoding is used.


In [6]:
encoding = tiktoken.get_encoding("cl100k_base")

sample_tokens = encoding.encode(X_train.iloc[0])

print("Number of tokens:", len(sample_tokens))
print("First token IDs:", sample_tokens[:30])


Number of tokens: 154
First token IDs: [72, 12756, 29395, 449, 279, 1732, 889, 1176, 29786, 922, 420, 5818, 13, 311, 757, 11, 420, 51728, 374, 922, 264, 893, 449, 6650, 74218, 19823, 4560, 813, 1888, 311]


## Rule 5  Pad or truncate

All reviews are changed to 200 tokens.


In [7]:
MAX_LENGTH = 200
PAD_TOKEN = encoding.n_vocab

def fix_length(ids):
    if len(ids) > MAX_LENGTH:
        return ids[:MAX_LENGTH]
    return ids + [PAD_TOKEN] * (MAX_LENGTH - len(ids))

def encode_reviews(reviews):
    return [
        fix_length(encoding.encode(review))
        for review in reviews
    ]

X_train_ids = encode_reviews(X_train)
X_test_ids = encode_reviews(X_test)

print("Train sequences:", len(X_train_ids))
print("Test sequences :", len(X_test_ids))
print("Sequence length:", len(X_train_ids[0]))
print("Padding ID:", PAD_TOKEN)


Train sequences: 8000
Test sequences : 2000
Sequence length: 200
Padding ID: 100277


## Rule 6  Convert sequences and labels to PyTorch tensors


In [8]:
X_train_tensor = torch.tensor(X_train_ids, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_ids, dtype=torch.long)

y_train_tensor = torch.tensor(
    y_train.to_numpy(),
    dtype=torch.float32
)
y_test_tensor = torch.tensor(
    y_test.to_numpy(),
    dtype=torch.float32
)

print("X_train:", X_train_tensor.shape)
print("y_train:", y_train_tensor.shape)
print("X_test :", X_test_tensor.shape)
print("y_test :", y_test_tensor.shape)


X_train: torch.Size([8000, 200])
y_train: torch.Size([8000])
X_test : torch.Size([2000, 200])
y_test : torch.Size([2000])


# Activity 1 RNN

## Rule 7  Embedding → nn.RNN → Linear


In [9]:
class RNNClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(
            encoding.n_vocab + 1,
            128,
            padding_idx=PAD_TOKEN
        )

        self.rnn = nn.RNN(
            128,
            96,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(192, 1)

    def forward(self, x):
        x = self.embedding(x)
        _, hidden = self.rnn(x)

        hidden = torch.cat(
            (hidden[0], hidden[1]),
            dim=1
        )

        return self.fc(hidden).squeeze(1)


rnn_model = RNNClassifier().to(device)

print(rnn_model)
print("Output shape:", rnn_model(
    X_train_tensor[:8].to(device)
).shape)


RNNClassifier(
  (embedding): Embedding(100278, 128, padding_idx=100277)
  (rnn): RNN(128, 96, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=192, out_features=1, bias=True)
)
Output shape: torch.Size([8])


## Rule 8 — Train the RNN


In [10]:
train_loader = DataLoader(
    TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=64,
    shuffle=True
)

criterion = nn.BCEWithLogitsLoss()

rnn_optimizer = torch.optim.Adam(
    rnn_model.parameters(),
    lr=0.0005,
    weight_decay=1e-5
)

RNN_EPOCHS = 12
rnn_losses = []

for epoch in range(RNN_EPOCHS):

    rnn_model.train()
    total_loss = 0.0

    for batch_x, batch_y in train_loader:

        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        output = rnn_model(batch_x)
        loss = criterion(output, batch_y)

        rnn_optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            rnn_model.parameters(), 1.0
        )

        rnn_optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    rnn_losses.append(avg_loss)

    print(
        f"Epoch [{epoch+1:02d}/{RNN_EPOCHS}] "
        f"Loss: {avg_loss:.4f}"
    )


Epoch [01/12] Loss: 0.6965
Epoch [02/12] Loss: 0.6699
Epoch [03/12] Loss: 0.6416
Epoch [04/12] Loss: 0.5864
Epoch [05/12] Loss: 0.5318
Epoch [06/12] Loss: 0.4849
Epoch [07/12] Loss: 0.4249
Epoch [08/12] Loss: 0.3817
Epoch [09/12] Loss: 0.3274
Epoch [10/12] Loss: 0.2725
Epoch [11/12] Loss: 0.2654
Epoch [12/12] Loss: 0.1950


## Rule 9  Test the RNN


In [11]:
rnn_model.eval()

with torch.no_grad():
    output = rnn_model(X_test_tensor.to(device))
    probability = torch.sigmoid(output)
    prediction = (probability >= 0.5).long()

actual = y_test_tensor.to(device).long()

correct = (prediction == actual).sum().item()
rnn_accuracy = correct / len(y_test_tensor) * 100

print("Correct predictions:", correct)
print("Total test samples:", len(y_test_tensor))
print(f"RNN Test Accuracy: {rnn_accuracy:.2f}%")


Correct predictions: 1339
Total test samples: 2000
RNN Test Accuracy: 66.95%


## Rule 10  Test the RNN on new reviews


In [12]:
new_reviews = [
    "This movie was absolutely fantastic and I really enjoyed it.",
    "The movie was boring and disappointing. I did not enjoy it.",
    "The acting was excellent and the story was very interesting."
]

def predict_rnn(review):
    ids = fix_length(encoding.encode(review))

    x = torch.tensor(
        [ids],
        dtype=torch.long,
        device=device
    )

    rnn_model.eval()

    with torch.no_grad():
        probability = torch.sigmoid(
            rnn_model(x)
        ).item()

    sentiment = "positive" if probability >= 0.5 else "negative"
    return sentiment, probability


rnn_custom = []

for review in new_reviews:
    sentiment, probability = predict_rnn(review)
    rnn_custom.append(sentiment)

    print("Review:", review)
    print("Predicted sentiment:", sentiment)
    print(f"Positive probability: {probability:.4f}")
    print("-" * 60)


Review: This movie was absolutely fantastic and I really enjoyed it.
Predicted sentiment: positive
Positive probability: 0.9722
------------------------------------------------------------
Review: The movie was boring and disappointing. I did not enjoy it.
Predicted sentiment: negative
Positive probability: 0.3197
------------------------------------------------------------
Review: The acting was excellent and the story was very interesting.
Predicted sentiment: positive
Positive probability: 0.8445
------------------------------------------------------------


# Activity 2  LSTM

The same processed dataset is reused. The recurrent layer is changed from RNN to LSTM.


## Rule 7 — Embedding → nn.LSTM → Linear


In [13]:
class LSTMClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(
            encoding.n_vocab + 1,
            128,
            padding_idx=PAD_TOKEN
        )

        self.lstm = nn.LSTM(
            128,
            96,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(192, 1)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)

        hidden = torch.cat(
            (hidden[0], hidden[1]),
            dim=1
        )

        return self.fc(hidden).squeeze(1)


lstm_model = LSTMClassifier().to(device)

print(lstm_model)
print("Output shape:", lstm_model(
    X_train_tensor[:8].to(device)
).shape)


LSTMClassifier(
  (embedding): Embedding(100278, 128, padding_idx=100277)
  (lstm): LSTM(128, 96, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=192, out_features=1, bias=True)
)
Output shape: torch.Size([8])


## Rule 8  Train the LSTM


In [14]:
lstm_optimizer = torch.optim.Adam(
    lstm_model.parameters(),
    lr=0.0005,
    weight_decay=1e-5
)

LSTM_EPOCHS = 12
lstm_losses = []

for epoch in range(LSTM_EPOCHS):

    lstm_model.train()
    total_loss = 0.0

    for batch_x, batch_y in train_loader:

        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        output = lstm_model(batch_x)
        loss = criterion(output, batch_y)

        lstm_optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            lstm_model.parameters(), 1.0
        )

        lstm_optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    lstm_losses.append(avg_loss)

    print(
        f"Epoch [{epoch+1:02d}/{LSTM_EPOCHS}] "
        f"Loss: {avg_loss:.4f}"
    )


Epoch [01/12] Loss: 0.6903
Epoch [02/12] Loss: 0.6600
Epoch [03/12] Loss: 0.5971
Epoch [04/12] Loss: 0.4937
Epoch [05/12] Loss: 0.4121
Epoch [06/12] Loss: 0.3462
Epoch [07/12] Loss: 0.2880
Epoch [08/12] Loss: 0.2398
Epoch [09/12] Loss: 0.1907
Epoch [10/12] Loss: 0.1508
Epoch [11/12] Loss: 0.1143
Epoch [12/12] Loss: 0.0978


## Rule 9  Test the LSTM


In [15]:
lstm_model.eval()

with torch.no_grad():
    output = lstm_model(X_test_tensor.to(device))
    probability = torch.sigmoid(output)
    prediction = (probability >= 0.5).long()

correct_lstm = (prediction == actual).sum().item()
lstm_accuracy = correct_lstm / len(y_test_tensor) * 100

print("Correct predictions:", correct_lstm)
print("Total test samples:", len(y_test_tensor))
print(f"LSTM Test Accuracy: {lstm_accuracy:.2f}%")


Correct predictions: 1499
Total test samples: 2000
LSTM Test Accuracy: 74.95%


## Rule 10 Test the LSTM on new reviews


In [16]:
def predict_lstm(review):
    ids = fix_length(encoding.encode(review))

    x = torch.tensor(
        [ids],
        dtype=torch.long,
        device=device
    )

    lstm_model.eval()

    with torch.no_grad():
        probability = torch.sigmoid(
            lstm_model(x)
        ).item()

    sentiment = "positive" if probability >= 0.5 else "negative"
    return sentiment, probability


lstm_custom = []

for review in new_reviews:
    sentiment, probability = predict_lstm(review)
    lstm_custom.append(sentiment)

    print("Review:", review)
    print("Predicted sentiment:", sentiment)
    print(f"Positive probability: {probability:.4f}")
    print("-" * 60)


Review: This movie was absolutely fantastic and I really enjoyed it.
Predicted sentiment: positive
Positive probability: 0.9995
------------------------------------------------------------
Review: The movie was boring and disappointing. I did not enjoy it.
Predicted sentiment: negative
Positive probability: 0.1044
------------------------------------------------------------
Review: The acting was excellent and the story was very interesting.
Predicted sentiment: positive
Positive probability: 0.9929
------------------------------------------------------------


# RNN vs LSTM  Final comparison


In [17]:
print("=" * 55)
print("RNN vs LSTM")
print("=" * 55)

print(f"RNN  Test Accuracy : {rnn_accuracy:.2f}%")
print(f"LSTM Test Accuracy : {lstm_accuracy:.2f}%")

if lstm_accuracy > rnn_accuracy:
    print("Higher test accuracy: LSTM")
elif rnn_accuracy > lstm_accuracy:
    print("Higher test accuracy: RNN")
else:
    print("Both models have the same accuracy.")

print("\nCustom review comparison:")

for i in range(len(new_reviews)):
    result = (
        "Agree"
        if rnn_custom[i] == lstm_custom[i]
        else "Different"
    )

    print(
        f"{i+1}. RNN: {rnn_custom[i]} | "
        f"LSTM: {lstm_custom[i]} | {result}"
    )

print("\nFinal training loss:")
print(f"RNN  : {rnn_losses[-1]:.4f}")
print(f"LSTM : {lstm_losses[-1]:.4f}")


RNN vs LSTM
RNN  Test Accuracy : 66.95%
LSTM Test Accuracy : 74.95%
Higher test accuracy: LSTM

Custom review comparison:
1. RNN: positive | LSTM: positive | Agree
2. RNN: negative | LSTM: negative | Agree
3. RNN: positive | LSTM: positive | Agree

Final training loss:
RNN  : 0.1950
LSTM : 0.0978
